# Draw — Treinamento SegFormer-B0 (4 classes: background / paper / hand / ink)

**O que este notebook faz:** treinar, no Kaggle com GPU, o modelo de
segmentação do Draw, exportar para ONNX e gerar os artefatos de entrega.

**O que este notebook NÃO faz (fora do escopo, Codex depois):**
webcam, integração no app, tracking temporal, memória de traços,
oclusão em tempo real, interface, CAD, simulador.

**O treino completo NÃO foi executado na máquina local.**
Lá não há GPU nem torch instalado. O smoke test local (ver `training/train.py --smoke`)
cobre config, pareamento, pesos, métricas e o contrato ONNX — mas o
forward pass com torch só é provado aqui, no Kaggle.

## Passo a passo no Kaggle

1. **GPU:** Settings do notebook -> Accelerator -> GPU (T4 ou P100).
2. **Inputs (Add Input):**
   - o dataset sintético gerado (outro agente): pastas
     `dataset/synthetic/{train,val,test}/{images,masks}` + `split_manifest.json`;
   - o FreiHAND (`danieldelro/freihand`) — só para a checagem de formato
     abaixo; o treino usa o dataset sintético, que já incorpora as mãos.
3. **Código:** este notebook espera os arquivos `training/` do repo.
   Ou clone a branch `feature/neural-seg-v1` do `GUZZBR1/draw`, ou suba
   `training/train.py` + `training/config.json` como dataset do Kaggle.
4. **Rode as células em ordem.** O treino salva tudo em `/kaggle/working/out`.
5. **Tempo estimado:** 30 épocas, ~5-10 mil imagens 512x512, batch 8:
   3-6 h em T4. Dataset menor (2 mil) ou 15 épocas: 1-2 h.
   O tempo por época é registrado no log; ajuste `--epochs` se preciso.
6. **Entrega:** baixe de `/kaggle/working/out`:
   `draw_segmentation_v1.onnx`, `classes.json`, `metrics.json`,
   `training_config.json`, `split_manifest.json`, `checkpoints/best.pt`.


## 1. Instalar dependências

torch, transformers (SegFormer), albumentations, onnx e onnxruntime.
O repo do Draw em si só precisa de numpy + opencv; o treino precisa
destas a mais, e elas vivem só no ambiente do Kaggle.


In [ ]:
!pip install -q torch transformers albumentations onnx onnxruntime
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Localizar código e dataset

Ajuste `REPO` e `DATA` se os caminhos do Kaggle forem outros.
O dataset sintético já vem com splits por fonte original (mão, folha,
fundo) — nunca reembaralhe fontes entre train/val/test aqui.


In [ ]:
import os, json, glob
from pathlib import Path

REPO = Path("/kaggle/working/draw")  # clone da branch feature/neural-seg-v1
if not (REPO / "training" / "train.py").exists():
    REPO = Path("/kaggle/working")  # fallback: training/ subido direto
print("repo:", REPO, "| train.py:", (REPO / "training" / "train.py").exists())

# Dataset sintético: primeiro input do Kaggle que contenha dataset/synthetic
cands = [p for p in Path("/kaggle/input").glob("*/dataset/synthetic")]
DATA = cands[0] if cands else Path("/kaggle/input/dataset/synthetic")
print("dataset sintético:", DATA)
for split in ("train", "val", "test"):
    n = len(list((DATA / split / "images").glob("*"))) if (DATA / split / "images").exists() else 0
    print(f"  {split}: {n} imagens")


## 3. Checagem do FreiHAND (não inventar rótulos)

Antes do treino, confirme o que a versão do Kaggle realmente contém:
imagens RGB utilizáveis? máscaras reais de mão? formato? quantidade?
resolução? licença? **Se não houver máscaras adequadas, pare aqui,**
documente o problema e troque a fonte — não gere rótulos de mão
sintéticos fingindo que são reais. O treino em si usa o dataset
sintético; esta célula é só a verificação exigida pelo escopo.


In [ ]:
from pathlib import Path
fh = list(Path("/kaggle/input").glob("*freihand*")) + list(Path("/kaggle/input").glob("*FreiHAND*"))
print("inputs FreiHAND encontrados:", [str(p) for p in fh])
for root in fh:
    for ext in ("*.png", "*.jpg", "*.json", "*.txt", "*.md", "*.zip"):
        files = list(root.rglob(ext))[:5]
        if files:
            print(f"  {ext}: {len(list(root.rglob(ext)))} arquivos, ex.: {files[0]}")
print("Se a lista acima estiver vazia ou sem máscaras de mão, NÃO prossiga sem documentar.")


## 4. Config e contagem de pixels

Lê `training/config.json` e conta pixels por classe no treino.
Compare com os pesos da config (CE [1, 2, 3, 10]): se a contagem real
divergir muito, recalibre os pesos e registre a mudança em
`training_config.json` — nunca ajustando peso olhando o split de teste.


In [ ]:
import sys, json
sys.path.insert(0, str(REPO))
from training.train import load_config, pair_images_masks, pixel_counts, compute_class_weights

cfg = load_config(REPO / "training" / "config.json")
print("modelo:", cfg["model"]["architecture"], cfg["model"]["backbone"])
print("classes:", cfg["model"]["classes"])
print("loss:", cfg["loss"]["formula"])
pairs = pair_images_masks(DATA / "train")
counts = pixel_counts(pairs)
print("pixels por classe (treino):", counts.tolist())
print("pesos calculados (referência):", [round(w, 2) for w in compute_class_weights(counts)])
print("pesos da config (usados):", cfg["loss"]["cross_entropy_class_weights"])


## 5. Treino

Chama `training/train.py`, que é a única fonte de verdade do treino
(modelo, loss, métricas, ONNX). O notebook é só um invólucro
reproduzível em cima dele. Em caso de OOM, o script sugere batch 4
com acumulação — não reduza a resolução: traço fino depende dela.


In [ ]:
!python {REPO}/training/train.py --config {REPO}/training/config.json --data-dir {DATA} --out-dir /kaggle/working/out


## 6. Métricas e cenários fixos

`metrics.json` traz IoU, Dice, precision e recall por classe, a
métrica de ink com tolerância (2-3 px) e a validação ONNX.
**Nunca reporte só accuracy global:** com fundo dominante, ela mente.
Os 15 cenários fixos (folha limpa, mão cobrindo canto/centro/traço,
sombra forte, traço fino de lápis, etc.) têm predições salvas para
comparação entre versões — o avaliador de cenários é o próximo passo
do agente de validação, fora deste notebook.


In [ ]:
import json
m = json.load(open("/kaggle/working/out/metrics.json"))
print("val_mean_iou:", round(m["mean_iou"], 4))
for cls in ("background", "paper", "hand", "ink"):
    d = m[cls]
    print(f"{cls:11s} IoU {d['iou']:.3f}  Dice {d['dice']:.3f}  P {d['precision']:.3f}  R {d['recall']:.3f}")
print("validação ONNX:", m.get("onnx_validation"))
print("artefatos:", sorted(p.name for p in __import__("pathlib").Path("/kaggle/working/out").iterdir()))


## 7. Interface ONNX (para o Codex integrar em `draw/neural.py`)

- **Input:** `image`, tensor float32 `[1, 3, 512, 512]`, RGB,
  normalização ImageNet (mean 0.485/0.456/0.406, std 0.229/0.224/0.225)
  — idêntica ao contrato atual de `draw/neural.py`.
- **Output:** `logits`, `[1, 4, 512, 512]`, sem softmax embutido.
  Ordem das classes: 0=background, 1=paper, 2=hand, 3=ink.
- **Uso:** `paper_mask = (argmax == 1)`, `hand_mask = (argmax == 2)`,
  `ink_mask = (argmax == 3)`.
- **paper_presence:** DERIVADO da máscara (fração de pixels paper
  acima do limiar que a integração escolher), não uma cabeça auxiliar.
  O modelo antigo tinha 2 canais + `presence_logit`; o novo tem 4
  canais e cabeça única — a integração muda só a leitura da saída,
  não o pré-processamento.
- **Regra de rótulo:** mão sobre traço = hand. O modelo não enxerga
  através da mão; a memória temporal futura do Draw é que preservará
  o traço escondido, usando `ink_full_mask` (fora do treino) como
  referência de avaliação.
